# 02 - Modèle Probabiliste Heston avec NumPyro

Ce notebook présente la définition du modèle probabiliste Heston avec NumPyro.

## Objectifs

1. Définir les priors pour les paramètres (κ, θ, σ, ρ, v0, μ)
2. Implémenter la vraisemblance du modèle Heston
3. Utiliser NumPyro pour définir le modèle
4. Visualiser les priors
5. Tester le modèle avec des données synthétiques

## 1. Importation des Bibliothèques

In [1]:
import sys
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import seaborn as sns
import numpyro
import numpyro.distributions as dist
from numpyro.infer import Predictive

# Configuration du style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Import du modèle Heston
sys.path.append('../src')
from models.heston_model import HestonModel, HestonModelConditional
from simulation.heston_sim import HestonSimulator

# Configuration de NumPyro
numpyro.set_platform('cpu')  # Utiliser CPU pour la compatibilité
numpyro.set_host_device_count(1)  # Une seule chaîne pour le moment

ImportError: cannot import name 'HestonModelConditional' from 'models.heston_model' (c:\Users\Benoi\Finance\2026-ECE-Ing4-Fin-IA-Projet2-Gr03\2026-ECE-Ing4-Fin-IA-Projet2-Gr03\A.4 Modélisation de volatilté stochastique (Heston\SABR) avec MCMC\notebooks\../src\models\heston_model.py)

## 2. Définition des Priors pour les Paramètres

Le modèle Heston a 5 paramètres principaux :

| Paramètre | Description | Prior | Justification |
|-----------|-------------|-------|---------------|
| **κ** | Vitesse de retour à la moyenne | LogNormal(log(2.0), 0.3) | Positif, centré sur 2 |
| **θ** | Variance de long terme | LogNormal(log(0.04), 0.3) | Positif, centré sur 0.04 |
| **σ** | Volatilité de la variance | LogNormal(log(0.3), 0.3) | Positif, centré sur 0.3 |
| **ρ** | Corrélation prix-volatilité | TruncatedNormal(-0.7, 0.2, -0.99, 0.99) | Borné, centré sur -0.7 |
| **v0** | Variance initiale | LogNormal(log(0.04), 0.3) | Positif, centré sur 0.04 |
| **μ** | Taux de rendement espéré | Normal(0.05, 0.1) | Centré sur 5% annuel |

In [ ]:
# Création du modèle Heston (approche par défaut basée sur les moments)
heston_model = HestonModel(
    S0=100.0,
    dt=1/252,
    mu=None,  # μ sera estimé
    use_moments_likelihood=True  # Vraisemblance basée sur les moments
)

print("Modèle Heston créé avec succès!")
print(f"  S0: {heston_model.S0}")
print(f"  dt: {heston_model.dt}")
print(f"  mu: {heston_model.mu}")
print(f"  use_moments_likelihood: {heston_model.use_moments_likelihood}")

## 3. Visualisation des Priors

In [ ]:
# Fonction pour générer des échantillons à partir des priors
def get_prior_samples_from_model(model, rng_key, n_samples=10000):
    """Génère des échantillons à partir des priors du modèle."""
    # Créer un modèle de prédiction sans données
    predictive = Predictive(model.model, num_samples=n_samples)
    
    # Données factices (ne sont pas utilisées pour les priors)
    dummy_returns = jnp.zeros((1, 10))
    
    # Générer les échantillons
    samples = predictive(rng_key, dummy_returns)
    
    # Extraire les paramètres transformés
    prior_samples = {
        'kappa': jnp.exp(samples['log_kappa']),
        'theta': jnp.exp(samples['log_theta']),
        'sigma': jnp.exp(samples['log_sigma']),
        'v0': jnp.exp(samples['log_v0']),
        'rho': samples['rho'],
    }
    
    if 'mu' in samples:
        prior_samples['mu'] = samples['mu']
    
    return prior_samples

# Fonction pour calculer le résumé des priors
def get_prior_summary(prior_samples):
    """Calcule le résumé statistique des priors."""
    summary = {}
    for param_name, samples in prior_samples.items():
        samples_np = np.array(samples)
        summary[param_name] = {
            'mean': np.mean(samples_np),
            'std': np.std(samples_np),
            'q2.5': np.percentile(samples_np, 2.5),
            'q25': np.percentile(samples_np, 25),
            'q50': np.percentile(samples_np, 50),
            'q75': np.percentile(samples_np, 75),
            'q97.5': np.percentile(samples_np, 97.5)
        }
    return summary

# Génération d'échantillons à partir des priors
rng_key = jax.random.PRNGKey(42)
n_samples = 10000

prior_samples = get_prior_samples_from_model(heston_model, rng_key, n_samples)

# Résumé des priors
prior_summary = get_prior_summary(prior_samples)

print("Résumé des Priors:")
print("=" * 60)
for param_name, stats in prior_summary.items():
    print(f"\n{param_name}:")
    print(f"  Moyenne: {stats['mean']:.4f}")
    print(f"  Écart-type: {stats['std']:.4f}")
    print(f"  2.5%: {stats['q2.5']:.4f}")
    print(f"  25%: {stats['q25']:.4f}")
    print(f"  50%: {stats['q50']:.4f}")
    print(f"  75%: {stats['q75']:.4f}")
    print(f"  97.5%: {stats['q97.5']:.4f}")

In [ ]:
# Visualisation des distributions des priors
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Paramètres à visualiser
params_to_plot = ['kappa', 'theta', 'sigma', 'rho', 'v0', 'mu']
param_labels = {
    'kappa': 'κ (Vitesse de retour à la moyenne)',
    'theta': 'θ (Variance de long terme)',
    'sigma': 'σ (Volatilité de la variance)',
    'rho': 'ρ (Corrélation)',
    'v0': 'v0 (Variance initiale)',
    'mu': 'μ (Taux de rendement)'
}

for idx, param_name in enumerate(params_to_plot):
    if param_name in prior_samples:
        samples = np.array(prior_samples[param_name])
        
        # Histogramme
        axes[idx].hist(samples, bins=50, density=True, alpha=0.7, edgecolor='black')
        axes[idx].axvline(x=np.mean(samples), color='r', linestyle='--', 
                          label=f'Moyenne = {np.mean(samples):.4f}')
        axes[idx].axvline(x=np.median(samples), color='g', linestyle='--',
                          label=f'Médiane = {np.median(samples):.4f}')
        axes[idx].set_xlabel(param_labels[param_name])
        axes[idx].set_ylabel('Densité')
        axes[idx].set_title(f'Prior: {param_name}')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Contraintes sur les Paramètres

In [ ]:
# Affichage des contraintes sur les paramètres
constraints = {
    'kappa': {
        'description': 'Vitesse de retour à la moyenne',
        'min': 0.0,
        'max': 10.0,
        'typical_range': (0.5, 5.0)
    },
    'theta': {
        'description': 'Variance de long terme',
        'min': 0.0,
        'max': 1.0,
        'typical_range': (0.01, 0.1)
    },
    'sigma': {
        'description': 'Volatilité de la variance',
        'min': 0.0,
        'max': 2.0,
        'typical_range': (0.1, 0.5)
    },
    'rho': {
        'description': 'Corrélation prix-volatilité',
        'min': -1.0,
        'max': 1.0,
        'typical_range': (-0.9, -0.2)
    },
    'v0': {
        'description': 'Variance initiale',
        'min': 0.0,
        'max': 1.0,
        'typical_range': (0.01, 0.1)
    },
    'mu': {
        'description': 'Taux de rendement espéré',
        'min': -1.0,
        'max': 1.0,
        'typical_range': (-0.1, 0.2)
    }
}

print("Contraintes sur les Paramètres:")
print("=" * 80)
for param_name, param_constraints in constraints.items():
    print(f"\n{param_name} - {param_constraints['description']}")
    print(f"  Min: {param_constraints['min']}")
    print(f"  Max: {param_constraints['max']}")
    print(f"  Plage typique: {param_constraints['typical_range']}")

## 5. Génération de Données Synthétiques pour le Test

In [ ]:
# Paramètres du modèle Heston pour la simulation
heston_params = {
    'S0': 100.0,
    'v0': 0.04,
    'mu': 0.05,
    'kappa': 2.0,
    'theta': 0.04,
    'sigma': 0.3,
    'rho': -0.7,
    'T': 1.0,
    'dt': 1/252,
    'seed': 42
}

# Simulation
simulator = HestonSimulator(**heston_params)
S, v, t = simulator.simulate(n_paths=1)

# Calcul des rendements
returns = simulator.get_returns()

print("Données synthétiques générées:")
print(f"  Nombre d'observations: {len(returns)}")
print(f"  Prix initial: {S[0, 0]:.4f}")
print(f"  Prix final: {S[-1, 0]:.4f}")
print(f"  Rendement total: {(S[-1, 0] / S[0, 0] - 1) * 100:.2f}%")
print(f"  Moyenne des rendements: {np.mean(returns):.6f}")
print(f"  Écart-type des rendements: {np.std(returns):.6f}")

## 6. Test du Modèle Probabiliste

In [ ]:
# Test du modèle avec les données synthétiques
print("Test du modèle probabiliste Heston...")

# Conversion en JAX array
returns_jax = jnp.array(returns.T)  # Shape: (1, n_obs)

# Test du modèle (vérification que le modèle s'exécute sans erreur)
try:
    # Création d'un modèle de test
    test_model = HestonModel(S0=heston_params['S0'], dt=heston_params['dt'])
    
    # Test avec numpyro.handlers.seed
    with numpyro.handlers.seed(rng_seed=42):
        test_model.model(returns_jax)
    
    print("✓ Modèle exécuté avec succès!")
except Exception as e:
    print(f"✗ Erreur lors de l'exécution du modèle: {e}")

## 7. Prior Predictive Check

In [ ]:
# Prior predictive check
print("Prior Predictive Check...")

# Génération de prédictions à partir des priors
rng_key, rng_key_pred = jax.random.split(rng_key)
prior_predictive = Predictive(heston_model.model, num_samples=100)
prior_pred = prior_predictive(rng_key_pred, returns_jax)

print(f"  Nombre d'échantillons: {prior_pred['log_kappa'].shape[0]}")
print(f"  Clés disponibles: {list(prior_pred.keys())}")

# Visualisation des paramètres générés
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, param_name in enumerate(params_to_plot):
    if param_name in prior_pred:
        samples = np.array(prior_pred[param_name])
        
        # Histogramme
        axes[idx].hist(samples, bins=50, density=True, alpha=0.7, edgecolor='black')
        axes[idx].axvline(x=np.mean(samples), color='r', linestyle='--', 
                          label=f'Moyenne = {np.mean(samples):.4f}')
        axes[idx].set_xlabel(param_labels[param_name])
        axes[idx].set_ylabel('Densité')
        axes[idx].set_title(f'Prior Predictive: {param_name}')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Comparaison des Approches de Vraisemblance

In [ ]:
# Création des deux modèles
model_moments = HestonModel(
    S0=heston_params['S0'],
    dt=heston_params['dt'],
    mu=None,
    use_moments_likelihood=True
)

model_conditional = HestonModelConditional(
    S0=heston_params['S0'],
    dt=heston_params['dt'],
    mu=None
)

print("Modèles créés:")
print(f"  - HestonModel (moments): use_moments_likelihood = {model_moments.use_moments_likelihood}")
print(f"  - HestonModelConditional: vraisemblance gaussienne conditionnelle")

# Test des deux modèles
print("\nTest des modèles...")
try:
    with numpyro.handlers.seed(rng_seed=42):
        model_moments.model(returns_jax)
    print("✓ HestonModel (moments) exécuté avec succès!")
except Exception as e:
    print(f"✗ Erreur HestonModel (moments): {e}")

try:
    with numpyro.handlers.seed(rng_seed=42):
        model_conditional.model(returns_jax)
    print("✓ HestonModelConditional exécuté avec succès!")
except Exception as e:
    print(f"✗ Erreur HestonModelConditional: {e}")

## 9. Visualisation des Moments Empiriques vs Théoriques

In [ ]:
# Calcul des moments empiriques des rendements
empirical_mean = np.mean(returns)
empirical_var = np.var(returns)
empirical_skew = np.mean((returns - empirical_mean)**3) / (np.std(returns)**3 + 1e-10)
empirical_kurt = np.mean((returns - empirical_mean)**4) / (np.std(returns)**4 + 1e-10)

print("Moments empiriques des rendements:")
print(f"  Moyenne: {empirical_mean:.6f}")
print(f"  Variance: {empirical_var:.6f}")
print(f"  Skewness: {empirical_skew:.4f}")
print(f"  Kurtosis: {empirical_kurt:.4f}")

# Moments théoriques du modèle Heston (avec les vrais paramètres)
dt = heston_params['dt']
theta = heston_params['theta']
sigma = heston_params['sigma']
rho = heston_params['rho']
mu = heston_params['mu']

theoretical_mean = mu * dt - 0.5 * theta * dt
theoretical_var = theta * dt
theoretical_skew = rho * sigma * np.sqrt(dt) / (np.sqrt(theta) + 1e-10)
theoretical_kurt = 3.0 + (sigma**2 * dt) / (theta + 1e-10)

print("\nMoments théoriques du modèle Heston:")
print(f"  Moyenne: {theoretical_mean:.6f}")
print(f"  Variance: {theoretical_var:.6f}")
print(f"  Skewness: {theoretical_skew:.4f}")
print(f"  Kurtosis: {theoretical_kurt:.4f}")

# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

moments = [
    ('Moyenne', empirical_mean, theoretical_mean),
    ('Variance', empirical_var, theoretical_var),
    ('Skewness', empirical_skew, theoretical_skew),
    ('Kurtosis', empirical_kurt, theoretical_kurt)
]

for idx, (name, emp, theo) in enumerate(moments):
    axes[idx].bar(['Empirique', 'Théorique'], [emp, theo], 
                  color=['blue', 'red'], alpha=0.7)
    axes[idx].set_ylabel('Valeur')
    axes[idx].set_title(f'{name} des Rendements')
    axes[idx].grid(True, alpha=0.3)
    
    # Ajouter les valeurs sur les barres
    axes[idx].text(0, emp, f'{emp:.4f}', ha='center', va='bottom')
    axes[idx].text(1, theo, f'{theo:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 10. Conclusion

Dans ce notebook, nous avons :

1. ✓ Défini les priors pour les paramètres (κ, θ, σ, ρ, v0, μ)
2. ✓ Implémenté la vraisemblance du modèle Heston
3. ✓ Utilisé NumPyro pour définir le modèle
4. ✓ Visualisé les priors
5. ✓ Testé le modèle avec des données synthétiques
6. ✓ Comparé les approches de vraisemblance (moments vs conditionnelle)

### Points clés :

- **Priors** : Choisis pour être très informatifs basés sur les connaissances financières
- **Vraisemblance** : Deux approches disponibles (moments empiriques ou gaussienne conditionnelle)
- **NumPyro** : Framework de programmation probabiliste pour l'inférence
- **Reparamétrization** : Utilisation de log-transform pour les paramètres positifs

### Prochaines étapes :

- Implémenter l'inférence MCMC (NUTS)
- Estimer les paramètres sur les données synthétiques
- Analyser les diagnostics de convergence (R-hat, ESS)
- Comparer les paramètres estimés avec les vrais paramètres